# GPU-Accelerated Portfolio Optimizer

### What this notebook covers
1. Real market data — 750 trading days, 50 S&P 500 large-caps (Yahoo Finance)
2. Monte Carlo simulation of 50 000 random portfolios
3. Gradient-based Sharpe maximisation (Modern Portfolio Theory)
4. **Out-of-sample backtesting** — optimise on first 500 days, evaluate on unseen next 250 days
5. Rolling walk-forward validation with quarterly rebalancing

### Running on Google Colab
1. Upload the project folder to Google Drive
2. Mount Drive (first cell)
3. Point `PROJECT_ROOT` at the folder path
4. Run all cells (`Runtime → Run all`)

> **No GPU required** — the code uses PyTorch vectorised operations which run on CPU if no GPU is present.
> **C++ scorer** is optional; pure-Python fallback activates automatically.

In [ ]:
# ── 1. Detect environment & install dependencies ─────────────────────────────
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    # Mount Google Drive (comment out if project was cloned directly to /content)
    from google.colab import drive
    drive.mount('/content/drive')
    # !! Update this path to wherever you placed the project on Drive !!
    PROJECT_ROOT = '/content/drive/MyDrive/portfolio-optimizer'
else:
    # Local: assume notebook is in project root
    PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
    if not os.path.exists(os.path.join(PROJECT_ROOT, 'backend')):
        PROJECT_ROOT = os.getcwd()  # notebook is already in project root

BACKEND = os.path.join(PROJECT_ROOT, 'backend')
print(f'Project root : {PROJECT_ROOT}')
print(f'Backend path : {BACKEND}')
print(f'On Colab     : {ON_COLAB}')

In [ ]:
# ── 2. Install Python dependencies ────────────────────────────────────────────
import subprocess
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy', 'scipy', 'pandas', 'yfinance', 'scikit-learn', 'torch', 'matplotlib'
])
print('Dependencies installed.')

In [ ]:
# ── 3. Import backend modules ─────────────────────────────────────────────────
sys.path.insert(0, BACKEND)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from data_pipeline import fetch_market_data, compute_statistics, DEFAULT_TICKERS
from optimizer import monte_carlo_gpu, optimize_sharpe_gpu, compute_efficient_frontier
from backtest import run_backtest, rolling_backtest, _perf_stats

print('All modules loaded.')

In [ ]:
# ── 4. Load & inspect market data ─────────────────────────────────────────────
prices = fetch_market_data()   # uses CSV cache if present, else fetches from Yahoo Finance

print(f'Shape          : {prices.shape}  (trading days × tickers)')
print(f'Date range     : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Tickers        : {list(prices.columns[:10])} … ({prices.shape[1]} total)')
print(f'Missing values : {prices.isna().sum().sum()}')
print()
print('Latest prices (top 5 by closing price):')
latest = prices.iloc[-1].sort_values(ascending=False).head(5)
for t, p in latest.items():
    print(f'  {t:8s}  ${p:.2f}')

In [ ]:
# ── 5. Individual asset statistics ────────────────────────────────────────────
stats = compute_statistics(prices)
tickers = stats['tickers']
mu_a = stats['mean_returns'] * 252          # annualised return
vol_a = np.sqrt(np.diag(stats['cov_matrix']) * 252)  # annualised vol
sharpe_ind = (mu_a - 0.04) / vol_a

print(f'Annual return range : {mu_a.min():.1%} to {mu_a.max():.1%}')
print(f'Annual vol range    : {vol_a.min():.1%} to {vol_a.max():.1%}')
print(f'Individual Sharpe   : {sharpe_ind.min():.2f} to {sharpe_ind.max():.2f}')
print()
df_assets = pd.DataFrame({'Return': mu_a, 'Volatility': vol_a, 'Sharpe': sharpe_ind},
                          index=tickers).sort_values('Sharpe', ascending=False)
print('Top 10 assets by Sharpe ratio:')
print(df_assets.head(10).to_string(float_format=lambda x: f'{x:.3f}'))

In [ ]:
# ── 6. Monte Carlo simulation (50 000 random portfolios) ──────────────────────
mc = monte_carlo_gpu(stats['mean_returns'], stats['cov_matrix'], n_portfolios=50000)

print(f'Portfolios simulated : {mc["n_portfolios"]:,}')
print(f'Computation time     : {mc["computation_time"]:.3f}s')
print(f'Device               : {mc["device"]}')
print()
ms = mc['max_sharpe']
mv = mc['min_volatility']
print(f'Best Sharpe  : return={ms["return"]:.1%}  vol={ms["volatility"]:.1%}  sharpe={ms["sharpe"]:.3f}')
print(f'Min Vol      : return={mv["return"]:.1%}  vol={mv["volatility"]:.1%}  sharpe={mv["sharpe"]:.3f}')

In [ ]:
# ── 7. Efficient Frontier plot ─────────────────────────────────────────────────
n_scatter = min(5000, len(mc['all_returns']))
idx = np.random.choice(len(mc['all_returns']), n_scatter, replace=False)
r_sc = np.array(mc['all_returns'])[idx]
v_sc = np.array(mc['all_volatilities'])[idx]
s_sc = np.array(mc['all_sharpe'])[idx]

frontier = compute_efficient_frontier(stats['mean_returns'], stats['cov_matrix'])

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(v_sc, r_sc, c=s_sc, cmap='viridis', alpha=0.4, s=8, label='MC portfolios')
plt.colorbar(sc, ax=ax, label='Sharpe ratio')

# Efficient frontier
ef_v = np.array(frontier['volatilities'])
ef_r = np.array(frontier['returns'])
ax.plot(ef_v, ef_r, 'r-', linewidth=2, label='Efficient frontier', zorder=5)

# Special portfolios
ax.scatter(ms['volatility'], ms['return'], marker='*', s=300, c='gold', edgecolors='black', zorder=6, label=f'Max Sharpe ({ms["sharpe"]:.2f})')
ax.scatter(mv['volatility'], mv['return'], marker='D', s=150, c='red',  edgecolors='black', zorder=6, label=f'Min Vol')

ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Annual Volatility'); ax.set_ylabel('Annual Return')
ax.set_title('Efficient Frontier — 50 000 Monte Carlo Portfolios\n50 S&P 500 Large-Caps')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Gradient-based Sharpe optimisation ─────────────────────────────────────
opt = optimize_sharpe_gpu(stats['mean_returns'], stats['cov_matrix'], n_iterations=2000)

print(f'Optimal portfolio (in-sample, on ALL {len(prices)} days):')
print(f'  Annual return : {opt["return"]:.1%}')
print(f'  Annual vol    : {opt["volatility"]:.1%}')
print(f'  Sharpe ratio  : {opt["sharpe"]:.3f}')
print()
print('⚠  NOTE: Sharpe above is IN-SAMPLE (optimised on the same data used to measure it).')
print('   This will be much lower out-of-sample. See the backtest cells below.')
print()

top_holdings = sorted(zip(tickers, opt['weights']), key=lambda x: -x[1])[:10]
print('Top holdings:')
df_h = pd.DataFrame(top_holdings, columns=['Ticker', 'Weight'])
df_h['Weight'] = df_h['Weight'].map('{:.1%}'.format)
print(df_h.to_string(index=False))

In [ ]:
# ── 9. Out-of-sample backtest (single train/test split) ───────────────────────
#
# Optimise on first 500 days.  Evaluate on the next ~250 days (UNSEEN data).
# This is the honest measure of whether the optimizer actually helps.

bt = run_backtest(prices, train_days=500)
sp = bt['split']

print('Single Train/Test Split')
print(f"  Train : {sp['train_start']} → {sp['train_end']} ({sp['train_days']} days) — used for optimisation")
print(f"  Test  : {sp['test_start']} → {sp['test_end']} ({sp['test_days']} days) — NOT seen during optimisation")
print()

rows = [
    ['In-sample (OVERFIT)',     bt['in_sample']],
    ['Out-of-sample optimised', bt['out_of_sample']['optimized']],
    ['Out-of-sample eq-weight', bt['out_of_sample']['equal_weight']],
]
table = pd.DataFrame(
    [(label, f"{s['annual_return']:.1%}", f"{s['annual_volatility']:.1%}",
      f"{s['sharpe_ratio']:.3f}", f"{s['max_drawdown']:.1%}")
     for label, s in rows],
    columns=['Portfolio', 'Annual Ret', 'Annual Vol', 'Sharpe', 'Max DD']
)
print(table.to_string(index=False))
print(f"\nOut-of-sample Sharpe lift vs equal-weight: {bt['out_of_sample']['sharpe_lift']:+.3f}")

In [ ]:
# ── 10. Rolling walk-forward backtest (quarterly rebalancing) ─────────────────
#
# More rigorous: re-optimise every 63 trading days (~quarter).
# Combined out-of-sample curve stitches together all test windows.

rbt = rolling_backtest(prices, train_days=400, step_days=63)

print(f"Rolling Walk-Forward ({len(rbt['windows'])} quarterly windows)")
print()
for i, w in enumerate(rbt['windows'], 1):
    print(f"  Window {i}: train {w['train_start']}→{w['train_end']}  "
          f"test {w['test_start']}→{w['test_end']}  "
          f"train Sharpe={w['opt_sharpe_train']:.2f}")

c = rbt['combined_out_of_sample']
rows2 = [
    ('Optimised (rolling)',   c['optimized']),
    ('Equal-weight baseline', c['equal_weight']),
]
print()
table2 = pd.DataFrame(
    [(label, f"{s['cumulative_return']:.1%}", f"{s['annual_return']:.1%}",
      f"{s['annual_volatility']:.1%}", f"{s['sharpe_ratio']:.3f}", f"{s['max_drawdown']:.1%}")
     for label, s in rows2],
    columns=['Portfolio', 'Cum Ret', 'Annual Ret', 'Annual Vol', 'Sharpe', 'Max DD']
)
print(table2.to_string(index=False))

In [ ]:
# ── 11. Equity curves — rolling walk-forward ──────────────────────────────────
dates_rbt = pd.to_datetime(rbt['cumulative']['dates'])
cum_opt = rbt['cumulative']['optimized']
cum_ew  = rbt['cumulative']['equal_weight']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative return curves
ax = axes[0]
ax.plot(dates_rbt, [(x - 1) * 100 for x in cum_opt], label='Optimised (rolling)', linewidth=2)
ax.plot(dates_rbt, [(x - 1) * 100 for x in cum_ew],  label='Equal-weight',        linewidth=2, linestyle='--')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Cumulative Return (%)')
ax.set_title('Out-of-Sample Equity Curves\n(Rolling Walk-Forward, Quarterly Rebalancing)')
ax.legend(); ax.grid(alpha=0.3)

# Right: allocation bar for the last rebalance
ax2 = axes[1]
top_h = bt['top_holdings'][:10]  # from single-split optimisation
labels = [t for t, _ in top_h]
weights = [w * 100 for _, w in top_h]
bars = ax2.barh(labels[::-1], weights[::-1], color='steelblue')
ax2.set_xlabel('Portfolio Weight (%)')
ax2.set_title('Top 10 Holdings\n(Most Recent Optimisation)')
ax2.grid(axis='x', alpha=0.3)
for bar, w in zip(bars, weights[::-1]):
    ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
             f'{w:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 12. Summary ───────────────────────────────────────────────────────────────
c_opt = rbt['combined_out_of_sample']['optimized']
c_ew  = rbt['combined_out_of_sample']['equal_weight']
sharpe_lift_pct = (c_opt['sharpe_ratio'] - c_ew['sharpe_ratio']) / abs(c_ew['sharpe_ratio']) * 100

print('=' * 60)
print('  SUMMARY — Out-of-Sample Performance')
print('=' * 60)
print(f'  Data        : 750 trading days, 50 S&P 500 large-caps')
print(f'  Source      : Yahoo Finance (yfinance) — REAL prices')
print(f'  Method      : Mean-Variance Optimisation (Markowitz MPT)')
print(f'  Evaluation  : Rolling walk-forward, 5 quarterly windows')
print()
print(f'  Optimised portfolio   : {c_opt["annual_return"]:.1%} annual  |  Sharpe {c_opt["sharpe_ratio"]:.3f}  |  MaxDD {c_opt["max_drawdown"]:.1%}')
print(f'  Equal-weight baseline : {c_ew["annual_return"]:.1%} annual  |  Sharpe {c_ew["sharpe_ratio"]:.3f}  |  MaxDD {c_ew["max_drawdown"]:.1%}')
print()
print(f'  Sharpe improvement    : +{sharpe_lift_pct:.0f}%  ({c_opt["sharpe_ratio"]:.3f} vs {c_ew["sharpe_ratio"]:.3f})')
print()
print('  ⚠  Caveat: Past performance. Period covers a bull market.')
print('     Real out-of-sample Sharpe (~0.75) is much lower than the')
print('     in-sample optimised figure (~3.8), which is pure overfitting.')
print('=' * 60)